# Tick bars and custom aggregations

Raw market data arrives as a tick stream: one trade per row, with a price,
a volume, and a timestamp. This notebook shows how `resample` condenses that
stream into fixed-interval bars. Three forms of the `agg` parameter are covered:

- `agg="ohlcv"` -- the standard open/high/low/close/volume bar via a
  built-in two-column reducer.
- `agg="ohlcv2"` -- the same OHLC columns plus volume split into
  buyer-initiated and seller-initiated halves, using the `PosPart`/`NegPart`
  decomposition.
- `agg={name: reducer, ...}` -- a dict of arbitrary per-bar statistics,
  here bar skewness and bar trend slope via `ExpandingSkew` and
  `ExpandingSlope`.

All three forms return a labelled `Stream` whose columns are accessible by
name. Every operator is causal: bar values are computed from ticks already
seen, never from future ones.

Related reading: [`resample` reference](../functions_streams/resample.md),
[`ExpandingSkew`](../functions_expanding/ExpandingSkew.md),
[`PosPart`](../functions_math/PosPart.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from screamer import PosPart, NegPart, ExpandingSkew, ExpandingSlope
from screamer.streams import resample

In [ ]:
# Reproducible synthetic tick stream.
rng = np.random.default_rng(42)
n = 80

# Irregular integer timestamps: each tick arrives 1-2 time units after the previous.
timestamps = np.cumsum(rng.integers(1, 3, size=n)).astype(np.int64)

# Mid-price: slow random walk around 100.
price = 100.0 + np.cumsum(rng.standard_normal(n) * 0.3)

# Trade size: heavy-tailed via exponential.
volume = rng.exponential(scale=50.0, size=n)

# Signed volume: positive = buyer-initiated, negative = seller-initiated.
# Sign is drawn independently of size, following a simple microstructure model.
signed_volume = volume * rng.choice([-1.0, 1.0], size=n)

print(f"{n} ticks, index range: {timestamps[0]}..{timestamps[-1]}")
pd.DataFrame({
    "timestamp":     timestamps[:6],
    "price":         price[:6].round(3),
    "volume":        volume[:6].round(1),
    "signed_volume": signed_volume[:6].round(1),
}).set_index("timestamp")

## 1. OHLCV bars

`agg="ohlcv"` expects a `(T, 2)` input: column 0 is price, column 1 is
unsigned volume. Each bar collects the first, maximum, minimum, and last
price (open/high/low/close) and sums the volume. The result is a labelled
`Stream` with five named columns.

The returned `Stream` unpacks as `(values, index)` for backward-compatible
tuple unpacking. Column access by name (`bars["close"]`) reads a 1-D view.

In [ ]:
BAR_WIDTH = 20

ohlcv = resample(np.column_stack([price, volume]), timestamps,
                 every=BAR_WIDTH, agg="ohlcv")

print("columns:", ohlcv.columns)
pd.DataFrame(
    ohlcv.values,
    index=pd.Index(ohlcv.index, name="bar_open"),
    columns=list(ohlcv.columns),
).round(3)

In [ ]:
# Candlestick-style bar chart: OHLC range on top, volume below.
bar_idx = ohlcv.index
fig, (ax_price, ax_vol) = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                                        gridspec_kw={"height_ratios": [2, 1]})

for t, row in zip(bar_idx, ohlcv.values):
    o, h, l, c = row[:4]
    color = "steelblue" if c >= o else "crimson"
    ax_price.plot([t, t], [l, h], color="0.4", lw=1.5, zorder=1)
    ax_price.bar(t, c - o, bottom=o, width=BAR_WIDTH * 0.5,
                 color=color, alpha=0.8, zorder=2)

ax_price.set_ylabel("price")
ax_price.set_title(f"OHLCV bars  (every={BAR_WIDTH})")

ax_vol.bar(bar_idx, ohlcv["volume"], width=BAR_WIDTH * 0.5, color="0.6")
ax_vol.set_ylabel("volume")
ax_vol.set_xlabel("bar open index")

plt.tight_layout()

## 2. Buy/sell volume decomposition with `ohlcv2`

`agg="ohlcv2"` takes the same two-column input but expects *signed* volume
in column 1 (positive = buyer-initiated, negative = seller-initiated). It
produces the same four OHLC columns plus two volume columns:

- `buy_vol`: sum of `max(signed_vol, 0)` per bar (= `sum(PosPart(signed_vol))`).
- `sell_vol`: sum of `max(-signed_vol, 0)` per bar (= `sum(NegPart(signed_vol))`).

The `PosPart`/`NegPart` pair is the signed decomposition: every real number
satisfies `x = PosPart(x) - NegPart(x)`. The cell below verifies that
`ohlcv2` buy/sell columns match explicit per-column resamples.

In [ ]:
ohlcv2 = resample(np.column_stack([price, signed_volume]), timestamps,
                  every=BAR_WIDTH, agg="ohlcv2")

print("columns:", ohlcv2.columns)
pd.DataFrame(
    ohlcv2.values,
    index=pd.Index(ohlcv2.index, name="bar_open"),
    columns=list(ohlcv2.columns),
).round(3)

In [ ]:
# Verify: buy_vol == sum(PosPart(signed_vol)), sell_vol == sum(NegPart(signed_vol)).
buy_check  = resample(np.asarray(PosPart()(signed_volume)), timestamps,
                      every=BAR_WIDTH, agg="sum")
sell_check = resample(np.asarray(NegPart()(signed_volume)), timestamps,
                      every=BAR_WIDTH, agg="sum")

np.testing.assert_allclose(ohlcv2["buy_vol"],  buy_check.values,  rtol=1e-12)
np.testing.assert_allclose(ohlcv2["sell_vol"], sell_check.values, rtol=1e-12)
print("PosPart / NegPart decomposition matches ohlcv2 buy_vol and sell_vol.")

# Net order flow per bar: positive means more buying than selling.
net_flow = ohlcv2["buy_vol"] - ohlcv2["sell_vol"]
pd.Series(net_flow.round(1), name="net_flow",
          index=pd.Index(ohlcv2.index, name="bar_open"))

## 3. Custom per-bar statistics with dict `agg`

Any `EvalOp` functor can serve as a bar reducer. The dict form runs several
reducers over the same bucketing and returns one labelled `Stream`. Each
functor is `reset()` automatically at every bar boundary, so bars accumulate
independently.

Two custom statistics are computed here:

- **`skew`**: intra-bar price skewness via `ExpandingSkew()`. It accumulates
  the first three central moments and returns the bias-corrected G1 coefficient
  at bar close. Defined once at least three ticks have been seen in the bar.
- **`slope`**: intra-bar price trend via `ExpandingSlope()`. It fits an OLS
  line to the price ticks against their within-bar position index (0, 1, 2, ...)
  and returns the slope at bar close. A positive slope means prices trended up
  inside the bar.

In [ ]:
custom = resample(
    price, timestamps, every=BAR_WIDTH,
    agg={"skew": ExpandingSkew(), "slope": ExpandingSlope()},
)

print("columns:", custom.columns)
pd.DataFrame(
    custom.values,
    index=pd.Index(custom.index, name="bar_open"),
    columns=list(custom.columns),
).round(4)

In [ ]:
# All three results share the same bar index, so we can join them.
close_series = pd.Series(
    ohlcv["close"],
    index=pd.Index(ohlcv.index, name="bar_open"),
    name="close",
)
custom_df = pd.DataFrame(
    custom.values,
    index=pd.Index(custom.index, name="bar_open"),
    columns=list(custom.columns),
)
combined = pd.concat([close_series, custom_df], axis=1)

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)

combined["close"].plot(ax=axes[0], marker="o", ms=5, color="steelblue")
axes[0].set_ylabel("close")
axes[0].set_title("Per-bar close price, intra-bar skewness, and intra-bar trend slope")

combined["skew"].plot(ax=axes[1], marker="o", ms=5, color="darkorange")
axes[1].axhline(0, color="0.6", lw=0.6)
axes[1].set_ylabel("skewness")

combined["slope"].plot(ax=axes[2], marker="o", ms=5, color="seagreen")
axes[2].axhline(0, color="0.6", lw=0.6)
axes[2].set_ylabel("slope")
axes[2].set_xlabel("bar open index")

plt.tight_layout()